> ☁️ **This notebook runs against the free shared sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")`. It uses features that need a server (fleet delegation, category allocations, shadow mode, or cross-process budgets). The sandbox is for evaluation only — **not for production** (may be wiped at any time; no SLA).
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - ☁️ **Free sandbox** _(this notebook)_ — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation.
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — your data, your infra; the production path for the server features this notebook uses. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
> - 💻 **Embedded** — `FiGuardClient()` — in-process, zero infra (single process; the server-only features used here aren't available embedded).

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")


In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")

# Orchestrator creates the parent fleet budget
fleet = client.create_budget(
    user_id=USER_ID,
    total_limit=1000.00,
    currency="USD",
    expires_in="24h",
    intent_context="multi-agent research and booking",
)

print(f"✓ Fleet budget created: {fleet.id}")
print(f"  Total limit: ${fleet.total_limit}")

# Issue scoped delegation tokens per sub-agent
research_token = client.create_delegation_token(
    budget_id=fleet.id,
    label="research_agent",
    caps=[{"category": "api", "limit": 200.00}],
)

booking_token = client.create_delegation_token(
    budget_id=fleet.id,
    label="booking_agent",
    caps=[{"category": "flight", "limit": 600.00}],
)

print(f"✓ research_agent cap: $200 (api)")
print(f"✓ booking_agent  cap: $600 (flight)")

In [ ]:
# research_agent spends against its delegation token
auth = client.authorize(
    session_token=research_token.session_token,
    agent_id="research_agent",
    action_type="EXTERNAL_CALL",
    description="Serper API: flight route research",
    requested_quantity=15.00,
    claimed_category="api",
    idempotency_key="research-001",
)
print(f"Research API call: {auth.decision} — ${auth.approved_quantity}")
client.confirm_event(auth.event_id, confirmed_quantity=15.00)
print("✓ Confirmed.")

# booking_agent spends against its delegation token
auth2 = client.authorize(
    session_token=booking_token.session_token,
    agent_id="booking_agent",
    action_type="PURCHASE",
    description="Delta JFK→LAX roundtrip",
    requested_quantity=450.00,
    claimed_category="flight",
    idempotency_key="booking-001",
)
print(f"Flight booking:    {auth2.decision} — ${auth2.approved_quantity}")
client.confirm_event(auth2.event_id, confirmed_quantity=450.00)
print("✓ Confirmed.")

# booking_agent tries to exceed its cap ($600 - $450 = $150 left)
auth3 = client.authorize(
    session_token=booking_token.session_token,
    agent_id="booking_agent",
    action_type="PURCHASE",
    description="Business class upgrade",
    requested_quantity=400.00,
    claimed_category="flight",
    idempotency_key="booking-002",
)
print(f"Upgrade attempt:   {auth3.decision} — {auth3.denial_reason}")
print("Delegation cap enforced. Fleet total unaffected.")

print(f"\n→ See the spend tree: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{fleet.id}")